# Water Loss Analytics
## 01 — Data Understanding
**Raden Satrio Hibatull Rasendriyo**

Saya menggunakan data audit kehilangan air dari California Department of Water Resources untuk mempelajari efisiensi sistem distribusi air. Pada notebook ini, saya memeriksa isi dan kualitas awal data sebelum melakukan cleaning dan analisis SQL.

Pertanyaan awal proyek:
1. Bagaimana cakupan laporan dan pola kehilangan air antarperiode?
2. Bagaimana membandingkan sistem yang ukuran layanannya berbeda?
3. Sistem mana yang perlu ditinjau lebih lanjut, dengan mempertimbangkan kehilangan air dan kualitas laporannya?

Pertanyaan tersebut masih perlu disesuaikan dengan hasil pemeriksaan data. Notebook ini belum memberikan peringkat kinerja atau kesimpulan tentang penyebab kehilangan air.


### Cara menjalankan
Simpan notebook ini di `02_Projects/water_loss_analytics/notebooks/`, lalu pilih kernel Python di VS Code dan jalankan **Run All**.

File yang dibutuhkan:
- `data/raw/ca-retailer-waterlossdata.csv`
- `data/reference/data_dictionary.csv`

Paket yang digunakan hanya **pandas** dan **IPython** (untuk menampilkan tabel). Jika muncul `ModuleNotFoundError`, jalankan `%pip install pandas ipython` di sel tambahan pada kernel yang sama, lalu restart kernel.

Notebook membaca data asli dan menyimpan ringkasan pemeriksaan ke `outputs/tables/`. File sumber tidak diubah. Ringkasan dengan nama yang sama akan diperbarui saat notebook dijalankan ulang.


### 1. Membaca data
Saya membaca semua kolom sebagai teks terlebih dahulu agar ID dan isi sumber tetap terjaga. Tanggal dan angka akan dikonversi pada salinan khusus untuk pemeriksaan.

Sumber: [California DWR — Annual Water Loss Audit Data](https://data.ca.gov/@california-department-of-water-resources/ca-urban-retail-water-system-water-loss-annual-loss-audit-data). Definisi kolom mengikuti `data_dictionary.csv` yang disertakan bersama dataset.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = []
for base in [cwd, *cwd.parents]:
    candidates.extend([base, base / 'water_loss_analytics',
                       base / '02_Projects' / 'water_loss_analytics'])
PROJECT_ROOT = next((p for p in candidates if
    (p / 'data/raw/ca-retailer-waterlossdata.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('CSV tidak ditemukan. Simpan notebook di water_loss_analytics/notebooks dan CSV di data/raw.')

raw_path = PROJECT_ROOT / 'data/raw/ca-retailer-waterlossdata.csv'
dict_path = PROJECT_ROOT / 'data/reference/data_dictionary.csv'
if not dict_path.is_file():
    raise FileNotFoundError(f'Kamus data tidak ditemukan: {dict_path}')

df_raw = pd.read_csv(raw_path, encoding='utf-8-sig', dtype='string')
data_dictionary = pd.read_csv(dict_path, encoding='utf-8-sig', dtype='string')
print(f'Folder proyek: {PROJECT_ROOT.name}')
print(f'Dataset: {len(df_raw):,} baris × {len(df_raw.columns):,} kolom')
print(f'Kamus data: {len(data_dictionary):,} baris definisi')
print(f'Versi pandas: {pd.__version__}')


Folder proyek: water_loss_analytics
Dataset: 4,936 baris × 190 kolom
Kamus data: 142 baris definisi
Versi pandas: 2.2.3


### 2. Memahami satu baris dan kolom utama
Satu baris merepresentasikan laporan audit suatu sistem air untuk satu periode. Satu organisasi dapat memiliki lebih dari satu sistem. Karena itu, jumlah baris tidak sama dengan jumlah perusahaan atau sistem unik.

Saya mulai dari identitas, periode, pasokan, serta komponen kehilangan air. Kolom kontak tidak ditampilkan karena tidak diperlukan untuk analisis ini.

| Kolom | Kegunaan |
|---|---|
| `DWR_ORGANIZATION_ID` | ID organisasi penyedia air |
| `PWS_ID_OR_OTHER_ID` | Identitas sistem pada laporan; formatnya perlu diperiksa |
| `REPORTING_START_DATE`, `REPORTING_END_DATE` | Batas periode laporan |
| `WS_WATER_SUPPLIED_VOL_AF` | Volume air yang dipasok, dalam acre-feet |
| `WL_REAL_LOSSES_VOL_AF` | Kehilangan fisik, misalnya kebocoran |
| `WL_APPARENT_LOSSES_VOL_AF` | Kehilangan semu, misalnya ketidakakuratan meter atau data |
| `NRW_NON_REVENUE_WATER_VOL_AF` | Air yang tidak menghasilkan pendapatan, termasuk konsumsi resmi yang tidak ditagihkan |
| `WATER_AUDIT_VALIDITY_SCORE` | Skor kualitas data audit; bukan jaminan semua angka benar |

Non-revenue water tidak sama dengan kebocoran fisik. Perbedaan definisi ini penting saat memilih metrik dashboard.


In [2]:
key_cols = ['DWR_ORGANIZATION_ID', 'WATER_SUPPLIER_NAME', 'PWS_ID_OR_OTHER_ID',
            'REPORTING_START_DATE', 'REPORTING_END_DATE', 'VOLUME_REPORTING_UNITS',
            'WS_WATER_SUPPLIED_VOL_AF', 'WL_REAL_LOSSES_VOL_AF',
            'WL_APPARENT_LOSSES_VOL_AF', 'NRW_NON_REVENUE_WATER_VOL_AF',
            'WATER_AUDIT_VALIDITY_SCORE', 'LATEST_STATUS']
missing_cols = sorted(set(key_cols) - set(df_raw.columns))
if missing_cols:
    raise ValueError(f'Kolom yang dibutuhkan tidak tersedia: {missing_cols}')
display(df_raw[key_cols].head())

# Salinan untuk pemeriksaan: spasi luar dan teks kosong diseragamkan.
# df_raw tetap dipertahankan.
df_check = df_raw.apply(lambda s: s.str.strip()).replace('', pd.NA)
print('ID organisasi unik:', df_check['DWR_ORGANIZATION_ID'].nunique())
print('Nilai ID sistem unik (belum dinormalisasi):', df_check['PWS_ID_OR_OTHER_ID'].nunique())


,DWR_ORGANIZATION_ID,WATER_SUPPLIER_NAME,PWS_ID_OR_OTHER_ID,REPORTING_START_DATE,REPORTING_END_DATE,VOLUME_REPORTING_UNITS,WS_WATER_SUPPLIED_VOL_AF,WL_REAL_LOSSES_VOL_AF,WL_APPARENT_LOSSES_VOL_AF,NRW_NON_REVENUE_WATER_VOL_AF,WATER_AUDIT_VALIDITY_SCORE,LATEST_STATUS
0,23,Alameda County Water District,CA0110001,2016-01-01,2016-12-31,Million gallons (US),37846.7985546577,1076.65456163467,914.986137482633,2044.05423470745,51.0,Requirements Addressed
1,23,Alameda County Water District,CA0110001,2017-01-01,2017-12-31,Million gallons (US),41086.5553913478,2432.33000948204,1116.98020784153,3656.35069010792,53.0,Requirements Addressed
2,23,Alameda County Water District,CA0110001,2018-01-01,2018-12-31,Million gallons (US),42093.0657410671,2630.97110505902,964.521070472566,3717.80411555063,56.0,Requirements Addressed
3,23,Alameda County Water District,CA0110001,2019-01-01,2019-12-31,Million gallons (US),41451.3477686275,2467.24710311517,948.84118799373,3529.41289233832,60.0,Requirements Addressed
4,23,Alameda County Water District,CA0110001,2020-01-01,2020-12-31,Million gallons (US),44396.5968885692,2470.78482878618,1044.56366780434,3657.02953117653,65.0,Requirements Addressed


ID organisasi unik: 440
Nilai ID sistem unik (belum dinormalisasi): 609


### 3. Memeriksa cakupan kamus data
Saya mencocokkan nama kolom CSV dengan nama kolom pada kamus. Kolom yang belum memiliki definisi tidak langsung saya tafsirkan berdasarkan namanya saja.


In [3]:
if 'Column' not in data_dictionary.columns:
    raise ValueError('Kamus data tidak memiliki kolom Column.')
defined_cols = set(data_dictionary['Column'].dropna().str.strip())
column_coverage = pd.DataFrame({'column': df_raw.columns})
column_coverage['documented'] = column_coverage['column'].isin(defined_cols)
print('Kolom CSV yang tercantum dalam kamus:', int(column_coverage['documented'].sum()))
print('Kolom CSV yang belum tercantum:', int((~column_coverage['documented']).sum()))
display(data_dictionary.loc[data_dictionary['Column'].str.strip().isin(key_cols),
                            ['Column', 'Label', 'Description']])


Kolom CSV yang tercantum dalam kamus: 140
Kolom CSV yang belum tercantum: 50


,Column,Label,Description
0,DWR_ORGANIZATION_ID,A CA DWR numeric identifier for each water sup...,Definition: A CA DWR numeric identifier for ea...
1,WATER_SUPPLIER_NAME,Name of CA urban retail water supplier,Definition: The name of the CA urban water sup...
13,REPORTING_START_DATE,Start date of the water loss audit (accounting...,Definition: Start of audit period explicity en...
14,REPORTING_END_DATE,End date of the water loss audit (accounting) ...,Definition: End of audit period explicity ente...
16,VOLUME_REPORTING_UNITS,Volume Reporting units,Definition: Volume Reporting units Source: Am...
17,PWS_ID_OR_OTHER_ID,Public water system identification number and ...,Definition: AWWA audit gives the option for th...
43,WS_WATER_SUPPLIED_VOL_AF,Total amount of water supplied by the water su...,Definition: Conversion of variable named WS_WA...
84,WL_APPARENT_LOSSES_VOL_AF,Estimated apparent water losses in acre-feet,Definition: Conversion of variable named WL_AP...
86,WL_REAL_LOSSES_VOL_AF,Estimated real water losses in acre-feet,Definition: Conversion of variable named WL_RE...
90,NRW_NON_REVENUE_WATER_VOL_AF,Conversion of variable NON_REVENUE_WATER_VOL ...,Definition: Conversion of variable NON_REVENU...


### 4. Kelengkapan data dan duplikasi
Nilai kosong belum tentu merupakan kesalahan. Beberapa kolom mungkin hanya berlaku pada versi formulir tertentu. Saya mencatatnya terlebih dahulu, tanpa menghapus baris atau mengisi nilai kosong dengan nol.

Untuk calon kunci laporan, saya gunakan gabungan ID organisasi, ID sistem, tanggal awal, dan tanggal akhir. Pemeriksaan ini belum membuktikan bahwa variasi penulisan ID sudah seragam.


In [4]:
missing_summary = pd.DataFrame({
    'missing_count': df_check.isna().sum(),
    'missing_pct': df_check.isna().mean().mul(100).round(2),
    'unique_non_null': df_check.nunique(dropna=True),
}).sort_values('missing_pct', ascending=False)
print('Kolom kosong seluruhnya:', int(missing_summary['missing_count'].eq(len(df_check)).sum()))
display(missing_summary.head(15))
print('Kelengkapan kolom utama:')
display(missing_summary.loc[key_cols])

key = ['DWR_ORGANIZATION_ID', 'PWS_ID_OR_OTHER_ID',
       'REPORTING_START_DATE', 'REPORTING_END_DATE']
complete_key = df_check[key].notna().all(axis=1)
duplicate_key = complete_key & df_check.duplicated(key, keep=False)
print('Baris duplikat penuh (salinan setelah kemunculan pertama):', int(df_raw.duplicated().sum()))
print('Baris dengan komponen kunci kosong:', int((~complete_key).sum()))
print('Baris yang terlibat duplikasi kunci lengkap:', int(duplicate_key.sum()))


Kolom kosong seluruhnya: 24


,missing_count,missing_pct,unique_non_null
REPORTING_YEAR,4936,100.0,0
SYSTEMAT_DATA_HANDLE_ERRORS_MT,4936,100.0,0
UNBILLED_UNMETERED_CONSUMP_MT,4936,100.0,0
UNBILLED_METERED_CONSUMP_MT,4936,100.0,0
WATER_EXPORTED_MT,4936,100.0,0
BILLED_METERED_CONSUMP_MT,4936,100.0,0
REAL_LOSSES_MT,4936,100.0,0
DIST_SYS_NOTES,4936,100.0,0
VOL_FROM_OWN_SOURCES_MT,4936,100.0,0
DISCRETE_DIST_NUMBER,4936,100.0,0


Kelengkapan kolom utama:


,missing_count,missing_pct,unique_non_null
DWR_ORGANIZATION_ID,2,0.04,440
WATER_SUPPLIER_NAME,0,0.00,446
PWS_ID_OR_OTHER_ID,0,0.00,609
REPORTING_START_DATE,0,0.00,33
REPORTING_END_DATE,0,0.00,36
VOLUME_REPORTING_UNITS,0,0.00,2
WS_WATER_SUPPLIED_VOL_AF,0,0.00,4896
WL_REAL_LOSSES_VOL_AF,0,0.00,4903
WL_APPARENT_LOSSES_VOL_AF,0,0.00,4897
NRW_NON_REVENUE_WATER_VOL_AF,0,0.00,4895


Baris duplikat penuh (salinan setelah kemunculan pertama): 0
Baris dengan komponen kunci kosong: 2
Baris yang terlibat duplikasi kunci lengkap: 0


### 5. Periode pelaporan
Kamus data menyarankan penggunaan tanggal awal dan akhir laporan daripada hanya mengandalkan `REPORTING_YEAR`. Saya memeriksa keberhasilan pembacaan tanggal dan panjang periodenya.

Tabel tahunan di bawah mengelompokkan laporan berdasarkan **tahun mulai**. Jumlah laporan yang lebih sedikit pada suatu tahun belum berarti kehilangan air menurun. Cakupan pelaporan dan perbedaan tahun fiskal perlu diperiksa terlebih dahulu.


In [5]:
dates = pd.DataFrame(index=df_check.index)
for col in ['REPORTING_START_DATE', 'REPORTING_END_DATE']:
    dates[col] = pd.to_datetime(df_check[col], format='%Y-%m-%d', errors='coerce')
    failed = df_check[col].notna() & dates[col].isna()
    print(f'{col}: kosong={df_check[col].isna().sum()}, gagal dibaca={failed.sum()}')

start = dates['REPORTING_START_DATE']
end = dates['REPORTING_END_DATE']
duration_days = (end - start).dt.days + 1
print('Rentang tanggal mulai:', start.min(), 'sampai', start.max())
print('Rentang tanggal akhir:', end.min(), 'sampai', end.max())
print('Periode dengan akhir sebelum awal:', int((end < start).sum()))
print('Periode di luar 365–366 hari (untuk ditinjau):',
      int((duration_days.notna() & ~duration_days.between(365, 366)).sum()))
print('REPORTING_YEAR terisi:', int(df_check['REPORTING_YEAR'].notna().sum()))

period_summary = pd.DataFrame({
    'start_year': start.dt.year.astype('Int64'),
    'calendar_year_period': start.dt.month.eq(1) & start.dt.day.eq(1)
        & end.dt.month.eq(12) & end.dt.day.eq(31) & start.dt.year.eq(end.dt.year)
}).groupby('start_year', dropna=False).agg(
    audit_count=('calendar_year_period', 'size'),
    calendar_year_count=('calendar_year_period', 'sum')
)
display(period_summary)
display(duration_days.describe().to_frame('duration_days'))


REPORTING_START_DATE: kosong=0, gagal dibaca=0
REPORTING_END_DATE: kosong=0, gagal dibaca=0
Rentang tanggal mulai: 2015-07-01 00:00:00 sampai 2025-01-01 00:00:00
Rentang tanggal akhir: 2016-06-30 00:00:00 sampai 2025-12-31 00:00:00
Periode dengan akhir sebelum awal: 0
Periode di luar 365–366 hari (untuk ditinjau): 27
REPORTING_YEAR terisi: 0


,audit_count,calendar_year_count
start_year,,
2015,6,0
2016,493,388
2017,531,421
2018,552,437
2019,558,439
2020,556,439
2021,564,431
2022,556,419
2023,549,416


,duration_days
count,4936.000000
mean,365.343801
std,1.981964
min,335.000000
25%,365.000000
50%,365.000000
75%,366.000000
max,426.000000


### 6. Satuan, status, dan versi formulir
Kolom volume mentah memakai satuan pelaporan yang berbeda. Untuk perbandingan volume, saya akan menggunakan kolom berakhiran `_AF` yang sudah dinyatakan dalam acre-feet.

Status laporan dan versi formulir juga perlu diperhatikan sebelum menentukan data yang masuk analisis utama. Semua status ditampilkan dahulu; belum ada penyaringan pada tahap ini.


In [6]:
category_tables = {}
for col in ['VOLUME_REPORTING_UNITS', 'LATEST_STATUS', 'UPLOAD_VERSION']:
    table = df_check[col].fillna('(kosong)').value_counts().rename_axis(col).reset_index(name='audit_count')
    category_tables[col] = table
    display(table)


,VOLUME_REPORTING_UNITS,audit_count
0,Acre-feet,3097
1,Million gallons (US),1839


,LATEST_STATUS,audit_count
0,Requirements Addressed,2369
1,Preliminarily Approved,2114
2,Requirements Not Addressed,310
3,Under Review,140
4,Not Required,3


,UPLOAD_VERSION,audit_count
0,V5,3171
1,V6.0,1645
2,V6.1,120


### 7. Memeriksa angka utama
Saya mengonversi kolom yang diperlukan pada salinan data. Tanda koma pemisah ribuan dibuang sebelum konversi. Nilai yang tidak terbaca dicatat, bukan dianggap nol.

Kehilangan air negatif perlu ditinjau karena dapat mencerminkan masalah estimasi atau pencatatan. Nilainya tidak saya hapus atau ubah otomatis. Skor validitas diperiksa pada rentang 0–100; skor tinggi tetap tidak menjamin setiap nilai benar.


In [7]:
numeric_cols = ['WS_WATER_SUPPLIED_VOL_AF', 'WL_REAL_LOSSES_VOL_AF',
                'WL_APPARENT_LOSSES_VOL_AF', 'NRW_NON_REVENUE_WATER_VOL_AF',
                'SD_NUM_SERVICE_CONN', 'SD_LENGTH_MAINS_MILES',
                'WATER_AUDIT_VALIDITY_SCORE']
numeric = pd.DataFrame(index=df_check.index)
checks = []
for col in numeric_cols:
    text_values = df_check[col].str.replace(',', '', regex=False)
    values = pd.to_numeric(text_values, errors='coerce')
    numeric[col] = values
    checks.append({
        'column': col, 'missing_source': int(df_check[col].isna().sum()),
        'parse_failed': int((df_check[col].notna() & values.isna()).sum()),
        'negative_count': int(values.lt(0).sum()), 'zero_count': int(values.eq(0).sum()),
        'min': values.min(), 'median': values.median(), 'max': values.max(),
    })
numeric_summary = pd.DataFrame(checks).set_index('column')
display(numeric_summary)
score = numeric['WATER_AUDIT_VALIDITY_SCORE']
print('Skor terisi di luar 0–100:', int((score.notna() & ~score.between(0, 100)).sum()))

# Contoh untuk ditinjau; tidak menampilkan informasi kontak.
negative_loss = numeric[['WL_REAL_LOSSES_VOL_AF', 'WL_APPARENT_LOSSES_VOL_AF',
                         'NRW_NON_REVENUE_WATER_VOL_AF']].lt(0).any(axis=1)
print('Baris dengan setidaknya satu komponen kehilangan/NRW negatif:', int(negative_loss.sum()))
display(df_check.loc[negative_loss, key_cols].head(10))


,missing_source,parse_failed,negative_count,zero_count,min,median,max
column,,,,,,,
WS_WATER_SUPPLIED_VOL_AF,0,0,0,0,1.252000,4149.898945,512337.773152
WL_REAL_LOSSES_VOL_AF,0,0,64,0,-2530.601720,199.574058,33035.711335
WL_APPARENT_LOSSES_VOL_AF,0,0,3,1,-166.841000,63.226198,8870.935197
NRW_NON_REVENUE_WATER_VOL_AF,0,0,25,0,-2253.740000,329.750386,49849.214768
SD_NUM_SERVICE_CONN,0,0,0,0,1.000000,7463.000000,755768.000000
SD_LENGTH_MAINS_MILES,0,0,0,0,0.162879,110.203835,7400.375000
WATER_AUDIT_VALIDITY_SCORE,0,0,0,0,26.000000,64.000000,94.000000


Skor terisi di luar 0–100: 0
Baris dengan setidaknya satu komponen kehilangan/NRW negatif: 67


,DWR_ORGANIZATION_ID,WATER_SUPPLIER_NAME,PWS_ID_OR_OTHER_ID,REPORTING_START_DATE,REPORTING_END_DATE,VOLUME_REPORTING_UNITS,WS_WATER_SUPPLIED_VOL_AF,WL_REAL_LOSSES_VOL_AF,WL_APPARENT_LOSSES_VOL_AF,NRW_NON_REVENUE_WATER_VOL_AF,WATER_AUDIT_VALIDITY_SCORE,LATEST_STATUS
76,55,Amador Water Agency,CA0300073,2017-01-01,2017-12-31,Acre-feet,2.24,-1.73472347597681e-16,0.0056,2.24,48.0,Requirements Addressed
77,55,Amador Water Agency,CA0300073,2018-01-01,2018-12-31,Acre-feet,3.13,-2.94902990916057e-17,0.007825,3.13,48.0,Requirements Addressed
78,55,Amador Water Agency,CA0300073,2019-01-01,2019-12-31,Acre-feet,2.39,-1.58727198051878e-16,0.005975,2.39,47.0,Requirements Addressed
113,55,Amador Water Agency,CA0310019,2018-01-01,2018-12-31,Acre-feet,16.94,-0.034962371134019,0.583212371134022,0.760000000000003,50.0,Requirements Addressed
115,55,Amador Water Agency,CA0310019,2020-01-01,2020-12-31,Acre-feet,17.86,-0.110518814432992,0.617268814432992,0.73,50.0,Requirements Not Addressed
351,816,El Dorado Irrigation District,CA0910017,2018-01-01,2018-12-31,Acre-feet,34.46,-0.660015736040615,0.70886573604061,0.134999999999995,58.0,Requirements Addressed
631,784,East Niles Community Services District,CA1510006,2018-01-01,2018-12-31,Acre-feet,7500.0,-240.374555837564,151.624555837564,5.0,60.0,Requirements Not Addressed
632,784,East Niles Community Services District,CA1510006,2019-01-01,2019-12-31,Acre-feet,6572.0,-724.973159898477,141.823159898477,-501.0,60.0,Requirements Not Addressed
694,2662,Wasco City Of,CA1510021,2021-01-01,2021-12-31,Million gallons (US),2966.693884595,-84.584071668708,75.4224659859695,-1.74051967207081,52.0,Requirements Not Addressed
733,123,Bakersfield City Of,CA1510031,2018-01-01,2018-12-31,Acre-feet,40201.78370048,-2004.93871343061,1041.22795465936,-863.206299520045,65.0,Requirements Not Addressed


### 8. Catatan sebelum cleaning
Dari pemeriksaan ini, saya perlu menetapkan beberapa aturan sebelum membuat grafik perbandingan:

- **Periode:** gunakan tanggal laporan dan pisahkan tahun kalender dari periode lain jika diperlukan. Perbandingan tren harus memperhatikan jumlah serta identitas sistem yang tersedia.
- **Satuan:** gunakan kolom volume `_AF` agar perbandingannya konsisten.
- **Kualitas data:** periksa nilai negatif, pasokan nol, kesalahan pembacaan angka, status laporan, dan skor validitas. Keputusan penyaringan harus dicatat beserta jumlah baris yang terdampak.
- **Ukuran sistem:** volume kehilangan air saja belum cukup untuk menyebut sebuah sistem tidak efisien. Metrik per sambungan atau per panjang jaringan perlu ditinjau beserta definisi dan kesesuaiannya.
- **Identitas:** nilai ID sistem unik belum otomatis menunjukkan jumlah sistem yang sudah terverifikasi; variasi format perlu diperiksa saat cleaning.

Saya belum memilih model machine learning pada tahap ini. Fokus berikutnya adalah menyusun data analisis yang dapat ditelusuri dan menghasilkan perbandingan yang masuk akal.


### 9. Menyimpan hasil pemeriksaan
Tabel berikut berisi ringkasan kualitas dan cakupan data. File ini bisa digunakan sebagai catatan saat menyusun notebook cleaning. Data mentah dan data bersih belum diekspor ulang.


In [8]:
output_dir = PROJECT_ROOT / 'outputs/tables'
output_dir.mkdir(parents=True, exist_ok=True)
exports = {
    '01_missing_summary.csv': missing_summary.rename_axis('column').reset_index(),
    '01_dictionary_coverage.csv': column_coverage,
    '01_period_summary.csv': period_summary.reset_index(),
    '01_numeric_checks.csv': numeric_summary.reset_index(),
    '01_status_summary.csv': category_tables['LATEST_STATUS'],
    '01_unit_summary.csv': category_tables['VOLUME_REPORTING_UNITS'],
    '01_form_version_summary.csv': category_tables['UPLOAD_VERSION'],
}
for filename, table in exports.items():
    table.to_csv(output_dir / filename, index=False, encoding='utf-8-sig')
    print(f'Tersimpan: outputs/tables/{filename}')
print('Pemeriksaan awal selesai. Data sumber tidak diubah.')


Tersimpan: outputs/tables/01_missing_summary.csv
Tersimpan: outputs/tables/01_dictionary_coverage.csv
Tersimpan: outputs/tables/01_period_summary.csv
Tersimpan: outputs/tables/01_numeric_checks.csv
Tersimpan: outputs/tables/01_status_summary.csv
Tersimpan: outputs/tables/01_unit_summary.csv
Tersimpan: outputs/tables/01_form_version_summary.csv
Pemeriksaan awal selesai. Data sumber tidak diubah.


### Langkah berikutnya
Lanjut ke `02_data_cleaning.ipynb` untuk menentukan kolom analisis, memeriksa format ID, menetapkan periode dan status yang digunakan, serta membuat penanda kualitas data. Hasil tahap ini menjadi dasar aturan cleaning, bukan alasan untuk langsung menghapus semua nilai yang terlihat tidak biasa.
